There are a few things you will need to change.

By running the next cell, you will be able to load a xlsx document with the data.

If your data has a different format, you should change a few things in the next cells.



In [ ]:
#[1] Access parameters
#The key is obtained from the AI/API Keys website.
#Important: Load credit in the Open AI account.
OPENAI_API_KEY_DO_NOT_SHARE = ''
#Using this field only if the key is associated to an organization in OpenAI. If they key is not associated to any specific organization the fielf can be empty.
OPENAI_ORGANIZATION = ''
 #This field is usefel if you manage multiple projects in the same organization. It is not necessary if you only manage the API with a standard account.
OPENAI_PROJECT_ID = ''

In [ ]:
#[2] Instalation of libraries to interact with the OpenAI API and manage text tokenization in Google Colab.
!pip install openai
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:00


In [ ]:
#[3] Importing several libraries to manage files, times, tokanization, JSON, API errors and retry strategies.
import pickle
import time
import tiktoken
import datetime
import json
import traceback
import numpy as np
from tqdm.notebook import tqdm
from jsonschema import validate
from openai import OpenAI, RateLimitError, APITimeoutError, InternalServerError, Timeout
from tenacity import retry, stop_after_attempt, wait_incrementing, retry_if_exception_type, after_log, before_sleep_log
import logging

In [ ]:
#[4] Defining rate and token limits
# ;ore info: https://platform.openai.com/docs/guides/rate-limits
rate_limit_per_minute = 1e3 # Max  1000 request per minute
token_limit_per_minute = 1e6  # Max 1,000,000 tokens per minute

#Calculate the minimal pause between solicitudes to avoid overpass the requests per minute limit (If the rate limit per minute = 1000, pause between each request will be 60 / 1000 = 0.06 seconds.
request_delay_seconds = 60.0 / rate_limit_per_minute  # Not change
request_timeout_seconds = 120   # Maximum time to wait before discarding a request
request_max_retries = 1         # Maximum number of retries on failure
tpm_wait_polling_seconds = 10   # Time to wait before checking if the token limit has been reset

# Global logger for static classes
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
#[5] Defining a ChatGPT class to interact with the OpenAI API, incorporating error handling, usage limits, and response validation.
class ChatGPT:
    def __init__(self, model, #Initializes a client to make requests to OpenAI.
                 halt_on_error=True,
                 is_verbose=True,
                 timeout=request_timeout_seconds,
                 max_retries=request_max_retries,
                 request_delay_seconds=request_delay_seconds,
                 token_limit_per_minute=token_limit_per_minute,
                 tpm_wait_polling_seconds=tpm_wait_polling_seconds,
                 logger=logger,
                 api_key=OPENAI_API_KEY_DO_NOT_SHARE,
                 organization=OPENAI_ORGANIZATION,
                 project=OPENAI_PROJECT_ID):
        self.model = model
        self.halt_on_error = halt_on_error
        self.is_verbose = is_verbose
        self.tpm_wait_polling_seconds = tpm_wait_polling_seconds
        self.request_delay_seconds = request_delay_seconds
        self.token_limit_per_minute = token_limit_per_minute
        self.response_history = []
        self.message_history = {}
        self.token_count_history = []
        self.logger = logger

        supported_models = ['gpt-4o-mini', 'gpt-4o-mini-2024-07-18', 'gpt-4o-2024-08-06']
        if self.model not in supported_models:
            raise Exception(f'{model} is not supported. Model must be one of {supported_models}')

        self.client = OpenAI(api_key = api_key,
                             organization = organization,
                             project = project,
                             timeout=timeout,
                             max_retries=max_retries)


    def get_running_cost_num_prompt_completion_tokens(self):  #Estimates the cost of tokens used in API requests.
        """
        This function computes the total cost (estimated) of all
        messages sent by the instance of ChatGPT called from
        Returns: total_running_cost, total_num_prompt_tokens, total_num_response_tokens
        """
        if self.model == 'gpt-4o-2024-08-06':
            prompt_cost = 0.0025 / 1000
            response_cost = 0.01 / 1000
        elif self.model == 'gpt-4o-mini':
            prompt_cost = 0.00015 / 1000
            response_cost = 0.0006 / 1000
        elif self.model == 'gpt-4o-mini-2024-07-18':
            prompt_cost = 0.00015 / 1000
            response_cost = 0.0006 / 1000
        else:
            prompt_cost = np.nan
            response_cost = np.nan

        n_prompt_tokens = np.sum([x.usage.prompt_tokens for x in self.response_history])
        n_completion_tokens = np.sum([x.usage.completion_tokens for x in self.response_history])
        return ((n_prompt_tokens * prompt_cost) + (n_completion_tokens * response_cost),
                n_prompt_tokens,
                n_completion_tokens)


    # Retry failing requests starting with 10 second wait,
    # Increasing wait time by 10 seconds each retry, up to a max window of 120s (or 5 times)
    # The goal is to try to avoid hitting backoff,
    # We treat this as a last resort because of its runtime cost
    @retry(wait=wait_incrementing(start=10, increment=10, max=120),
           stop=stop_after_attempt(5),
           retry=retry_if_exception_type((RateLimitError, APITimeoutError, InternalServerError, Timeout)),
           before_sleep=before_sleep_log(logger, logging.INFO),
           after=after_log(logger, logging.INFO))
    def completion_with_backoff(self, client, **kwargs):  #Use retry to retry requests in case of errors (e.g. usage limits).
        return client.chat.completions.create(**kwargs)


    def check_internal_TPM_tracker(self, n_message_tokens): #Control how many tokens have been sent in the last minute. If the limit is exceeded, wait before sending more requests.
        """
        Checks internal TPM count to see if a message with length = n_message_tokens
        can be sent. If not, it waits (sleeps - blocking) until the message delivery
        meets into TPM limit
        """
        now = datetime.datetime.now()
        one_minute_ago = now + datetime.timedelta(seconds=-60)
        self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
        n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        # Fixed delay waiting if TPM exceeded over past minute
        # This is cpu polling, so it doesnt cost money or much compute
        while n_tokens_past_minute > self.token_limit_per_minute:
            if self.is_verbose: self.logger.info(f'Internal TPM limit exceeded, waiting for {self.tpm_wait_polling_seconds} seconds...')
            time.sleep(self.tpm_wait_polling_seconds)
            now = datetime.datetime.now()
            one_minute_ago = now + datetime.timedelta(seconds=-60)
            self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
            n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        now = datetime.datetime.now()
        self.token_count_history.append((n_message_tokens, now))


    def send_message(self, system_role, message, json_schema): #Check that the request's json_schema contains certain required fields. Control the request and token limits per minute. Send the message to the OpenAI API with a conversation history. Handle errors and validate the API response.
        """
        This is the primary function used to send messages to GPT and get responses
        Steps are:
          - check that json schema meets our basic requirements
          - handle RPM and TPM limits as best as we can
            (when openai rejects requests for exceeding limits its much slower)
          - build and send the message using openai ChatCompletion api
          - perform basic validation on GPT's response
        This function either returns a ChatCompletion response object or None (if failure occurred)
        Errors are propogated using raised Exceptions
        """
        # Check json_schema fullfil minimun requeriments
        if 'refusal' not in json_schema['schema']['properties']:
            return self.bad_response_output(f'Output JSON schema must contain the field "refusal" under "properties". Missing in `json_schema` passed as input.')
        if 'reason_for_refusal' not in json_schema['schema']['properties']:
            return self.bad_response_output(f'Output JSON schema must contain the field "reason_for_refusal" under "properties". Missing in `json_schema` passed as input.')

        # Sleep based on RPM limit (lazy logic, avoids keeping a running count of actual requests per minute)
        time.sleep(self.request_delay_seconds)

        # Check the TPM limit (not lazy, uses continuous token count per minute)
        encoding = tiktoken.encoding_for_model(self.model)
        n_message_tokens = len(encoding.encode(system_role)) + len(encoding.encode(message))
        if n_message_tokens > self.token_limit_per_minute:
            return self.bad_response_output(f'Unable to send message as it exceeds TPM. Number of tokens in message = {n_message_tokens}')
        self.check_internal_TPM_tracker(n_message_tokens)

        # Build and send messages through OpenAI-API
        message_id = len(self.message_history.keys())
        self.message_history[message_id] = [{"role": "system", "content": system_role}]
        self.message_history[message_id].append({"role": "user", "content": message})
        try:
            response = self.completion_with_backoff(self.client,
                                                    model=self.model,
                                                    messages=self.message_history[message_id],
                                                    temperature=0,
                                                    stream=False,
                                                    response_format={"type": "json_schema",
                                                                     "json_schema": json_schema},
                                                    seed=0, logprobs=True)
        except Exception as e:
            if self.halt_on_error:
                raise
            else:
                if self.is_verbose:
                    str_e = str(e)
                    self.logger.info(f'An exception occurred: {str_e}')
                    self.logger.info(traceback.format_exc())
                return None

        self.response_history.append(response)

        # Response validation, check if GPT's finish_reason is valid
        finish_reason = response.choices[0].finish_reason
        if finish_reason != 'stop':
            return self.bad_response_output(f'Invalid GPT finish reason: {finish_reason}')

        # Response validation, check if json contents are valid
        if not self.json_response_validation(response, json_schema):
            return self.bad_response_output(f'GPT response failed validation')
        return (response, message_id)


    def bad_response_output(self, error): #Logs errors and decides whether to stop execution or continue.
        # General function to inform the user when an error occurs.
        if self.halt_on_error:
            raise Exception(error)
        else:
            if self.is_verbose:
                self.logger.info(f'Error - {error}')
        return None


    def json_response_validation(self, response, json_schema):#Checks that the OpenAI response is valid JSON and conforms to the expected schema. Rejects responses that the model refuses to respond to.
          # Basic request validation: All GPT responses are checked with this logic.
          # To validate additional data or responses, the user must define additional wrapper functions.
          # for the send_message function call.
        try:
            # Checks that it is a valid JSON string.
            response_json = json.loads(response.choices[0].message.content)
            # Checks that the JSON content matches the schema.
            validate(instance=response_json,
                     schema=json_schema['schema'])
            # Checks if the response is valid.
            if bool(response_json['refusal']):
                refusal_reason = response_json['reason_for_refusal']
                raise Exception(f'GPT refused to respond. Reason: "{refusal_reason}"')
            else:
                return True
        except Exception as e:
            if self.is_verbose:
                str_e = str(e)
                self.logger.info(f'Failed to validate response - {str_e}')
            return False

Now there are a few additional things you'll need to change.

By running the next cell, you will be able to load a xlsx document with the data.

If your data has a different format, you should change a few things in the next cells.


In [ ]:
#[6] Load a xlsx file
from google.colab import files
uploaded = files.upload()

Saving Goals - Dataframe (n150).xlsx to Goals - Dataframe (n150) (1).xlsx


In [ ]:
#[7] Read a xlsx file in the pandas dataframe. File name must be adjusted.
import pandas as pd
import io

#pd.read_excel(...) Read a xlsx file. Use pd.read_csv if the file has a csv format.
df = pd.read_excel(io.BytesIO(uploaded['Goals - Dataframe (n150) (1).xlsx']))
print(df)

            ResponseId                                               Goal
0    R_5qyRkMgVjPmYFuW  Basic description: 'focus on my store and my c...
1    R_1CJi5ERYKgccldv  Basic description: 'Better economic situation'...
2           3147299442  Basic description: 'Swim three times a week'. ...
3    R_1yp7uZe5UEf38yN  Basic description: 'independent business'. Add...
4           3168773261  Basic description: 'Exercise at least three ti...
..                 ...                                                ...
145         3022594882  Basic description: 'Perform academically in my...
146  R_3pLhPLlZj3yWxMt  Basic description: 'OWN HOUSE'. Additional inf...
147         3133843582  Basic description: 'Submit the thesis'. Additi...
148  R_7tb6quuJpxhpswM  Basic description: 'Invest'. Additional inform...
149  R_6ZhWVZACFV7qSJ3  Basic description: 'Own houses and properties....

[150 rows x 2 columns]


In [ ]:
#[8]Extract specific columns from a DataFrame (df) and converts them to a more manageable format for processing with GPT.
##In this case, only the ResponseID and the Goal are taken, which will be used to traverse the GPT calls
transcript_data = df[["ResponseId","Goal"]]
transcript_data = transcript_data.values
transcript_data

array([['R_5qyRkMgVjPmYFuW',
        "Basic description: 'focus on my store and my colorimetry courses'. Additional information: 'I am very focused on achieving this because in the past I managed to have a paint store in Venezuela. Today, I am in Colombia, a beautiful country, and here I want to achieve my goals since I have more than 20 years in the field of colorimetry'."],
       ['R_1CJi5ERYKgccldv',
        "Basic description: 'Better economic situation'. Additional information: 'This goal has been important for several years because I want to provide my family with a better quality of life. I hope to achieve it VERY soon'."],
       ['3147299442',
        "Basic description: 'Swim three times a week'. Additional information: 'This goal is important to me because it is part of a personal exercise to achieve discipline. Swimming is a sport I enjoy and it is also good for my physical and emotional health. I want to achieve it to feel stronger and more capable of reaching goals. For 

Configure the system role, message, and JSON schema for the responses.

In [ ]:
#[9] Define the system role
QA_system_role = ("You are an AI assistant helping a social researcher analyze personal goals.")

In [ ]:
#[10] Generate a message with embedded transcription data.
def generate_QA_user_message(transcript):
    transcript_json = {"transcript": transcript.strip()}
    transcript_json_str = json.dumps(transcript_json)

    message = """
Context
We are studying the pursuit and achievement of personal goals. Personal goals are defined as the outcomes a person aims to achieve, guided by their needs, interests, values, and aspirations.
We conducted a longitudinal study in which participants described their personal goals, and weeks later, we evaluated their progress. To analyze the kinds of goals people pursue, we need to classify the reported goals into predefined categories.
Each participant provided:
A basic description of their goal (what they want to achieve).
Additional information (why the goal is important, how long they have been pursuing it, and how they feel about it).
You will see a table containing information about personal goals from a group of individuals. The "goal" column presents the basic description of a goal along with any additional information they provided about its relevance.

Role
You are an AI assistant helping a social researcher analyze personal goals.

Task Definition
Your task is to classify and score each personal goal based on the following categories:
1. Education and Learning: Goals related to completing educational programs, acquiring new knowledge, abilities, or learning new skills (e.g., Finish my university studies because it is the career I chose and have always liked; Learn English to communicate my products to my audience and for job opportunities; learn how to drive).
2. Financial Stability and Independence: Goals focused on achieving financial security, independence, or improving economic conditions (e.g., Achieve financial stability to live more freely without worrying about money; Get out of debt to improve my economy and gain financial freedom).
3. Career and Professional Development: Goals related goals related to professional and work-life development. This would include job promotions, career changes, transitioning out of unemployment, and starting independent work through personal businesses or entrepreneurial ventures (e.g., Change jobs to experience new challenges and grow professionally;start my own business to provide more peace and well-being to my family).
4. Health and Well-being: Goals aimed at improving physical health, mental health, or overall well-being (e.g., Lose weight for health reasons and to feel strong for myself and my daughters; Exercise more to improve health and physical appearance).
5. Family and Relationships: Goals centered around improving family life, relationships, or supporting family members (e.g., Improve relationship with my son to maintain harmony and trust; Spend more quality time with my partner).
6. Travel and Exploration: Goals involving traveling to new places for leisure, cultural experiences, or personal fulfillment (e.g., Travel to Japan to learn about a new culture and see the world beyond my area; Travel around the world to achieve a higher status in my professional life).
7. Material Acquisition: Goals aimed at acquiring, retaining, or maintaining material goods. These may include purchasing a first or second home, vehicles, household appliances, leisure or personal enjoyment items, as well as home repairs or improvements (e.g.,Buy my own house; Buy a new car; Finish purchasing the furniture for my apartment.).
8. Personal Development and Fulfillment: Goals aimed at personal growth, self-improvement, or achieving a sense of fulfillment (e.g., To be a better person every day to feel happy and improve daily; Stop procrastinating to engage in activities that help me grow personally and professionally).
9. Not Applicable: Responses that are meaningless, nonsensical, or do not fit into any relevant category (e.g., Develop telepathy through meditation and practice; None, I don't have to give opinions about my goals).
10. Orphans: All goals that, although they make sense and are perfectly intelligible, do not fit into any of the categories.

Classification Rules
Each goal description can contain one or multiple goals. Classify the goals as follows:
Main Category (Score = 2): The ultimate goal the person wants to achieve.Each goal description must have one and only one main category.
Intermediate Category (Score = 1): Any additional goal that serves as a step toward achieving the main goal. A goal description may have one, multiple, or no intermediate categories.
Irrelevant Categories (Score = 0): Categories that do not apply to the goal description. Any category not assigned a 1 or 2 should be scored 0.

Example:
 For the goal "saving to buy a house", assign:
2 in the Material Acquisition category (ultimate goal: buying a house).
1 in the Financial Stability and Independence category (saving is a step toward buying a house).
0 in all other categories.

Whenever possible, classify the goal based only on the basic description. Use additional information only when the description is unclear, ambiguous, or unintelligible.

Expected Output
Present the responses in a tabular format.
Each goal description should have a score (0, 1, or 2) for each category in a separate column.

The diary transcript to be evaluated, provided in JSON format, is:

    """ + transcript_json_str

    return message

In [ ]:
#[11]Specify the format that the response should be in.
diary_assessment_response_json = {"name": "diary_assessment_response_json",
                                  "description": "Fetches diary transcript assessment response",
                                  "schema": {
                                      "description": "JSON schema for diary transcript assessment",
                                      "type": "object",
                                      "properties": {
                                          "statement_1_applies": {
                                              "type": "integer",
                                              "description": "How much does category 1 apply to the personal goal described."
                                          },
                                          "statement_2_applies": {
                                              "type": "integer",
                                              "description": "How much does category 2 apply to the personal goal described."
                                          },
                                          "statement_3_applies": {
                                              "type": "integer",
                                              "description": "How much does category 3 apply to the personal goal described."
                                          },
                                          "statement_4_applies": {
                                              "type": "integer",
                                              "description": "How much does category 4 apply to the personal goal described."
                                          },
                                          "statement_5_applies": {
                                              "type": "integer",
                                              "description": "How much does category 5 apply to the personal goal described."
                                          },
                                          "statement_6_applies": {
                                              "type": "integer",
                                              "description": "How much does category 6 apply to the personal goal described."
                                          },
                                          "statement_7_applies": {
                                              "type": "integer",
                                              "description": "How much does category 7 apply to the personal goal described."
                                          },
                                          "statement_8_applies": {
                                              "type": "integer",
                                              "description": "How much does category 8 apply to the personal goal described."
                                          },
                                          "statement_9_applies": {
                                              "type": "integer",
                                              "description": "How much does category 9 apply to the personal goal described."
                                          },
                                          "statement_10_applies": {
                                              "type": "integer",
                                              "description": "How much does category 1 apply to the personal goal described."
                                          },
                                          "refusal": {
                                              "type": "boolean",
                                              "description": "Indicates whether GPT refused to process the request."
                                          },
                                          "reason_for_refusal": {
                                              "type": "string",
                                              "description": "If the response was refused, this field must contain the reason for refusal."
                                          },
                                      },
                                      "required": [
                                          "statement_1_applies",
                                          "statement_2_applies",
                                          "statement_3_applies",
                                          "statement_4_applies",
                                          "statement_5_applies",
                                          "statement_6_applies",
                                          "statement_7_applies",
                                          "statement_8_applies",
                                          "statement_9_applies",
                                          "statement_10_applies",
                                          "refusal",
                                          "reason_for_refusal"
                                      ],
                                      "additionalProperties": False
                                  },
                                  "strict": True
                                 }


In [ ]:
# [12] Define the wrapper function to send messages to GPT and handle errors
def QA_messaging_wrapper(chat, system_role, message, json_schema):
    # with halt_on_error set to True in ChatGPT class,
    # we use exception propogation to handle errors and edge-cases
    message_history_id = -1
    statement_1_applies = None
    statement_2_applies = None
    statement_3_applies = None
    statement_4_applies = None
    statement_5_applies = None
    statement_6_applies = None
    statement_7_applies = None
    statement_8_applies = None
    statement_9_applies = None
    statement_10_applies = None
    try:
        response, message_history_id = chat.send_message(system_role=system_role,
                                                         message=message,
                                                         json_schema=json_schema)
        assert response is not None
        response_json = json.loads(response.choices[0].message.content)
        statement_1_applies = response_json["statement_1_applies"]
        statement_2_applies = response_json["statement_2_applies"]
        statement_3_applies = response_json["statement_3_applies"]
        statement_4_applies = response_json["statement_4_applies"]
        statement_5_applies = response_json["statement_5_applies"]
        statement_6_applies = response_json["statement_6_applies"]
        statement_7_applies = response_json["statement_7_applies"]
        statement_8_applies = response_json["statement_8_applies"]
        statement_9_applies = response_json["statement_9_applies"]
        statement_10_applies = response_json["statement_10_applies"]

    except Exception as e:
        response_json = {}
        response_str = ''
        chat.logger.info(f'Messaging wrapper failure - {str(e)}')

    cost, n_prompt_tokens, n_completion_tokens = chat.get_running_cost_num_prompt_completion_tokens()
    return (statement_1_applies, statement_2_applies, statement_3_applies, statement_4_applies,
            statement_5_applies, statement_6_applies, statement_7_applies, statement_8_applies,
            statement_9_applies, statement_10_applies), cost

Now we will actually set up the instance, selecting the GPT model here.


In [ ]:
#[13] Inicializating GPT
chat = ChatGPT(model='gpt-4o-2024-08-06')

This next section is the loop that loops over the rows in your input file, passes the story in to add to the prompt, and then gets a response. It prints out information showing the response was okay and a running cost estimate.

In [ ]:
#[14]Loop over lines in the file uploaded above, appended each transcript to the prompt, getting a response, and storing the responses in a list
rubric_GPT_responses = []
running_cost = 0
for i, (ID,transcript) in enumerate(tqdm(transcript_data)):
    qa_message = generate_QA_user_message(transcript)

    assessment, cost = QA_messaging_wrapper(chat, QA_system_role, qa_message, diary_assessment_response_json)
    rubric_GPT_responses.append([ID] + list(assessment))
    running_cost += cost
    print(f'({i+1}/{len(transcript_data)}) running cost: {running_cost}')

  0%|          | 0/150 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


(1/150) running cost: 2.8071225


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


(2/150) running cost: 5.618895


...

(148/150) running cost: 466.30092000000013


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


(149/150) running cost: 469.79918250000014


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


(150/150) running cost: 473.3020725000001


Finally, we take the responses and output them into a CSV file, which it will then download to your computer.

In [ ]:
#[15]Write a file with output
import pandas as pd

out_df = pd.DataFrame(rubric_GPT_responses)
out_df.columns = ['ResponseId',
                  'statement_1_applies',
                  'statement_2_applies',
                  'statement_3_applies',
                  'statement_4_applies',
                  'statement_5_applies',
                  'statement_6_applies',
                  'statement_7_applies',
                  'statement_8_applies',
                  'statement_9_applies',
                  'statement_10_applies']
out_df.head()

out_df.to_csv('responses.csv')
files.download('responses.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>